# Annexe · Tout refaire : le jeu, les entrées de 2018, l'apprentissage

Les carnets 01 à 08 lisent les résultats livrés (`../sorties/apprentissage`, `../modeles`). Ce
carnet refait tout, dans `../generes/` :

1. **le jeu** : 800 épisodes de 6 h (`conduites.grand_jeu`) ; environ 20 s par épisode et par cœur ;
   un épisode sur huit environ est abandonné (EPANET ne converge pas) ;
2. **les entrées de 2018**, jour par jour (`conduites.entrees2018_brutes`) ; environ 35 s par jour et
   par cœur ;
3. **l'apprentissage** des modèles choisis, puis leurs cartes de 2018 et le test simulé
   (`conduites.apprendre_gpu`) ; un GPU est conseillé. Les modèles appris en même temps tournent
   chacun dans leur processus (compter plusieurs Go de mémoire par modèle) ;
4. **les notes** de 2018 (`conduites.comparer2018`), avec les cartes du 08c livrées.

Il faut les mesures de BattLeDIM (voir `data/README.md` à la racine), fuites de 2018 comprises.

Avec les réglages stricts du prototype (EPANET s'arrête s'il ne converge pas), un épisode ou un
jour de 2018 qui ne converge pas est abandonné. Cela dépend du processeur : sur x86, environ 1
épisode sur 8 et aucun jour de 2018 ; sur un serveur ARM, mesuré 2 épisodes sur 6 et 3 jours sur
4 (jours 30 à 33) ; un Mac M1 à M4 est sans doute dans le même cas (non vérifié). Sur ARM, les notes de 2018 porteraient sur peu de jours.

**Trois modes** :
- `"rien"` : ne fait rien (par défaut) ;
- `"essai"` : 2 épisodes par partition, 2 jours de 2018, quelques minutes d'apprentissage de B. Une
  dizaine de minutes : cela vérifie que tout tourne, les notes n'ont pas de sens ;
- `"complet"` : la vraie expérience.

In [1]:
import shutil, subprocess, sys, time
from pathlib import Path

MODE = "rien"                         # "rien", "essai" ou "complet"
PROCESSUS = 4                         # cœurs pour fabriquer le jeu et les entrées de 2018
VARIANTES = "B,C"                     # ou "B,C,B_sans_residu_dual,B_instantane" (les retraits)
MINUTES = 105                         # durée de l'apprentissage en mode complet

ICI = Path("..").resolve()            # le dossier 09_conduites
G = ICI / "generes" / MODE

def lancer(*args):
    print("$ python -m", " ".join(map(str, args)), flush=True)
    t0 = time.time()
    subprocess.run([sys.executable, "-m", *map(str, args)], cwd=ICI, check=True)
    print(f"   fini en {time.time() - t0:.0f} s", flush=True)

## 1. Le jeu

In [2]:
if MODE == "essai" and G.exists():
    shutil.rmtree(G)                  # l'essai repart de zéro
if MODE == "rien":
    print("mode rien : on saute")
elif (G / "jeu" / "manifeste_0.json").exists():
    print("jeu déjà fait :", G / "jeu")
else:
    lancer("conduites.grand_jeu", "--sortie", G / "jeu", "--processus", PROCESSUS,
           *(["--par-partition", 2] if MODE == "essai" else []))

mode rien : on saute


## 2. Les entrées de 2018

In [3]:
if MODE == "rien":
    print("mode rien : on saute")
elif (G / "annee" / "schema.json").exists():
    print("entrées de 2018 déjà faites :", G / "annee")
else:
    lancer("conduites.entrees2018_brutes", "--sortie", G / "annee", "--processus", PROCESSUS,
           *(["--debut", 200, "--jours", 2] if MODE == "essai" else []))   # essai : 2 jours avec des fuites

mode rien : on saute


## 3. L'apprentissage

Le meilleur état (plus basse perte de validation) est gardé dans
`apprentissage/<modèle>/<objectif>_meilleur.pt`, au même format que les modèles livrés.

In [4]:
if MODE == "rien":
    print("mode rien : on saute")
else:
    lancer("conduites.apprendre_gpu", "--jeux", G / "jeu", "--annee", G / "annee",
           "--sortie", G / "apprentissage",
           "--variantes", "B" if MODE == "essai" else VARIANTES,
           "--minutes", 4 if MODE == "essai" else MINUTES,
           "--valider-min", 1 if MODE == "essai" else 8)

mode rien : on saute


## 4. Les notes de 2018

In [5]:
if MODE == "rien":
    print("mode rien : on saute")
else:
    shutil.copy(ICI / "sorties" / "apprentissage" / "cartes_08c_2018.npz", G / "apprentissage")
    lancer("conduites.comparer2018", "--sortie", G / "apprentissage")
    import pandas as pd
    t = pd.read_csv(G / "apprentissage" / "comparaison_2018.csv")
    display(t[["carte", "mode", "top-5 à 300 m", "empreinte 20%"]].round(3))

mode rien : on saute


Pour lire les nouveaux résultats dans les carnets 1 à 3 : copier le contenu de
`generes/complet/apprentissage/` dans `sorties/apprentissage/` (ils lisent `sorties/` et attendent
B et C), puis refaire `sorties/rejeu/` avec `python -m conduites.rejouer`.